In [1]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.feature_selection import SelectFromModel


from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import cross_val_score

df = pd.read_csv('datasets.csv')

df.replace(['?', '\t?'], np.nan, inplace=True)

num_cols = ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']
cat_cols = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane'] # 'rac' is often problematic, excluding for robustness

df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

# Define features (X) and target (y)
X = df.drop('class', axis=1)
y = df['class'].str.strip() # Clean target variable

drop_cols = ['hemo', 'sg', 'pcv', 'rbcc', 'al', 'rac']
X_reduced = X.drop(columns=drop_cols, errors='ignore')


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])


final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),

    ('selector', SelectFromModel(
        RandomForestClassifier(n_estimators=100, random_state=42),
        threshold='median' 
    )),

    ('classifier', LogisticRegression(
        C=0.1,
        max_iter=1000,
        solver='lbfgs',
        random_state=42
    ))
])


scores = cross_val_score(final_model, X, y, cv=5, scoring='accuracy')

print("--- Model Evaluation Results ---")
print(f"Individual cross-validation scores: {scores}")
print(f"Average accuracy: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")


from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score

final_model.fit(X, y)
train_preds = final_model.predict(X)
train_acc = accuracy_score(y, train_preds)

print(f"Training accuracy: {train_acc:.4f}")
print(f"Cross-validation accuracy (mean): {scores.mean():.4f}")

if train_acc - scores.mean() > 0.05:
    print("⚠️ Likely overfitting: training accuracy is much higher than CV accuracy.")
else:
    print("✅ Model generalizes well (no significant overfitting).")


--- Model Evaluation Results ---
Individual cross-validation scores: [1.         0.98571429 0.98571429 0.97142857 0.98571429]
Average accuracy: 0.9857 (+/- 0.0181)
Training accuracy: 0.9914
Cross-validation accuracy (mean): 0.9857
✅ Model generalizes well (no significant overfitting).


In [3]:
import pickle


In [5]:
file_path = 'kidney_model.pkl'
with open(file_path, 'wb') as file:
    pickle.dump(final_model, file)

print(f"Model successfully saved to {file_path}")

Model successfully saved to kidney_model.pkl
